In [ ]:
import os
import sys
import re
import json
from datetime import datetime, timedelta
from typing import List, Dict, Any
from openai import OpenAI
from langchain.tools.tavily_search import TavilySearchResults

# ====== paths for your project utils ======
sys.path.append("/Users/vaibhav/Langchain/Project Preparation/NewsSense")
from utils.excel_io import read_alerts, write_alerts

# ====== config ======
INPUT_PATH  = "/Users/vaibhav/Langchain/Project Preparation/NewsSense/data/dummy_alerts.xlsx"
OUTPUT_PATH = "/Users/vaibhav/Langchain/Project Preparation/NewsSense/data/alerts_output.xlsx"

# Optional synonyms mapping file (JSON): {"crude oil":["WTI","Brent"], "gold":["XAU","bullion"]}
ONTOLOGY_SYNONYMS_PATH = "/Users/vaibhav/Langchain/Project Preparation/NewsSense/data/ontology_synonyms.json"

# Toggle: use LLM to re-check relevance after filters (higher precision, slightly more cost)
USE_LLM_RERANK = True

# ====== clients ======
tavily = TavilySearchResults(max_results=12)
llm = OpenAI()

# ====== preferred sources ======
SOURCES_TIER1 = [
    "reuters.com", "bloomberg.com", "ft.com", "wsj.com",
    "apnews.com", "cnbc.com", "trading.com"
]
SOURCES_TIER2 = [
    "marketwatch.com", "moneycontrol.com", "economictimes.indiatimes.com",
    "seekingalpha.com", "nasdaq.com", "investing.com", "tradingview.com",
    "cnn.com", "bbc.com"
]

# ====== generic market-moving keywords (require at least one) ======
KEYWORDS = (
    "volatility|surge|spike|jump|plunge|sell-off|rally|production cut|sanction|"
    "strike|shutdown|OPEC|rate hike|inflation|CPI|war|conflict|embargo|"
    "inventory|draw|build|downgrade|ban|export|import|tariff|supply|demand|"
    "price|futures|options|hedge|flows|positions|positioning|output"
)

# ====== date patterns to extract pub date from snippet/URL ======
DATE_PATTERNS = [
    r'(\d{4})[-/](\d{1,2})[-/](\d{1,2})',            # 2025-08-07
    r'([A-Z][a-z]{2,9})\s+(\d{1,2}),\s+(\d{4})',     # August 7, 2025
    r'(\d{1,2})\s+([A-Z][a-z]{2,9})\s+(\d{4})',      # 07 August 2025
]

# =========================
# helpers
# =========================
def _load_synonyms(path: str) -> Dict[str, List[str]]:
    try:
        if path and os.path.exists(path):
            with open(path, "r") as f:
                data = json.load(f)
                # lowercase keys for robust lookup
                return {k.lower(): v for k, v in data.items() if isinstance(v, list)}
    except Exception:
        pass
    return {}

ONTOLOGY_SYNONYMS = _load_synonyms(ONTOLOGY_SYNONYMS_PATH)

def _site_filter(sites: List[str]) -> str:
    return " OR ".join([f"site:{s}" for s in sites])

def _mk_queries(ontology: str, alert_date_str: str, sites=None, strict=True) -> List[str]:
    dt = datetime.strptime(alert_date_str, "%d %B %Y")
    prev = dt - timedelta(days=1)
    d1 = dt.strftime("%d %B %Y")
    d0 = prev.strftime("%d %B %Y")
    base = f'{ontology} market news' if strict else f'{ontology}'
    if sites:
        base += f" ({_site_filter(sites)})"
    return [f'{base} "{d1}"', f'{base} "{d0}"']

def _dedupe(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen, out = set(), []
    for r in items:
        url = (r.get("url") or "").strip()
        if url and url not in seen:
            seen.add(url)
            out.append(r)
    return out

def _score(item: Dict[str, Any], ontology: str) -> int:
    text = f'{item.get("title","")} {item.get("content","")}'
    url  = (item.get("url") or "").lower()
    s = 0
    if ontology.lower() in text.lower(): s += 2
    if re.search(KEYWORDS, text, flags=re.I): s += 1
    if any(d in url for d in SOURCES_TIER1): s += 1
    s += min(len(text) // 300, 1)
    return s

def _try_parse_date(text: str):
    if not text: return None
    for pat in DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            try:
                if pat == DATE_PATTERNS[0]:
                    y, mo, d = map(int, m.groups()); return datetime(y, mo, d)
                elif pat == DATE_PATTERNS[1]:
                    mon, d, y = m.groups(); return datetime.strptime(f"{mon} {d} {y}", "%B %d %Y")
                else:
                    d, mon, y = m.groups(); return datetime.strptime(f"{d} {mon} {y}", "%d %B %Y")
            except Exception:
                continue
    return None

def _is_in_window(item_dt: datetime, target_dt: datetime) -> bool:
    if not item_dt: return False
    return item_dt.date() in {target_dt.date(), (target_dt - timedelta(days=1)).date()}

def _allow_patterns_for(ontology: str) -> List[re.Pattern]:
    """
    Build allow-patterns automatically from ontology + optional synonyms.
    If ontology is multi-word (e.g., "natural gas"), include the full phrase and each token.
    """
    terms = set()
    o = ontology.strip()
    if o: terms.add(o)
    syns = ONTOLOGY_SYNONYMS.get(o.lower(), [])
    for s in syns:
        if s: terms.add(s)

    # add tokens for multi-word ontologies
    more = set()
    for t in list(terms):
        toks = [w for w in re.split(r"\s+", t) if len(w) >= 3]
        more.update(toks)
    terms.update(more)

    # compile to regex with word boundaries, case-insensitive
    patterns = [re.compile(rf"\b{re.escape(t)}\b", re.I) for t in terms if t]
    # if no terms (shouldn’t happen), fallback to plain ontology word boundary
    if not patterns and o:
        patterns = [re.compile(rf"\b{re.escape(o)}\b", re.I)]
    return patterns

def _required_patterns() -> List[re.Pattern]:
    return [re.compile(KEYWORDS, re.I)]

def _filter_relevant(items: List[Dict[str, Any]], ontology: str, target_dt: datetime) -> List[Dict[str, Any]]:
    allow_patterns   = _allow_patterns_for(ontology)
    require_patterns = _required_patterns()

    out = []
    for it in items:
        title   = (it.get("title") or "").strip()
        snippet = (it.get("content") or "").strip()
        combined = f"{title} {snippet}".strip()
        url = it.get("url") or ""

        pub_dt = _try_parse_date(combined + " " + url)
        it["_pub_dt"] = pub_dt
        if not _is_in_window(pub_dt, target_dt):
            continue

        # Must match ontology (allow) in title/snippet
        if allow_patterns and not any(p.search(combined) for p in allow_patterns):
            continue

        # Must match at least one market-moving keyword
        if require_patterns and not any(p.search(combined) for p in require_patterns):
            continue

        out.append(it)
    return out

def _search_stage(ontology: str, alert_date: str, sites: List[str], tier_label: str, strict: bool) -> List[Dict[str, Any]]:
    raw = []
    for q in _mk_queries(ontology, alert_date, sites, strict=strict):
        out = tavily.run(q)  # List[{"content","url", maybe "title"}]
        if isinstance(out, list):
            for item in out:
                item["_tier"] = tier_label
            raw.extend(out)
    return raw

def _llm_rerank(items: List[Dict[str, Any]], ontology: str) -> List[Dict[str, Any]]:
    if not items:
        return items
    # compact list for LLM
    lines = []
    for i, it in enumerate(items, 1):
        title = it.get("title") or ""
        snippet = (it.get("content") or "").replace("\n", " ").strip()
        url = it.get("url") or ""
        if len(snippet) > 220: snippet = snippet[:217] + "…"
        lines.append(f"{i}. {title} — {snippet} ({url})")
    prompt = f"""
You are filtering market news for "{ontology}".
Keep only items that are clearly about this ontology AND describe market-moving factors
(volatility, price/volume moves, supply/demand, policy/geopolitics, OPEC/Fed, sanctions, inventories, safe-haven flows, etc.).
Return the kept item indices as a comma-separated list (e.g., "1,3,7"). If none are relevant, return "none".

News:
{chr(10).join(lines)}
"""
    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        max_tokens=50,
        messages=[
            {"role": "system", "content": "You precisely filter market-relevant news about a given ontology."},
            {"role": "user", "content": prompt},
        ],
    ).choices[0].message.content.strip().lower()

    if "none" in resp:
        return []
    kept = []
    for tok in re.split(r"[^\d]+", resp):
        if tok.isdigit():
            idx = int(tok) - 1
            if 0 <= idx < len(items):
                kept.append(items[idx])
    return kept

# =========================
# NEWS AGENT
# =========================
def run_news_agent(ontology: str, alert_date: str, k: int = 4) -> str:
    target_dt = datetime.strptime(alert_date, "%d %B %Y")

    # Pass 1: Tier1 strict
    raw = _search_stage(ontology, alert_date, SOURCES_TIER1, "Tier1-Strict", strict=True)
    items = _filter_relevant(_dedupe(raw), ontology, target_dt)

    # Pass 2: Tier1 loose
    if not items:
        raw += _search_stage(ontology, alert_date, SOURCES_TIER1, "Tier1-Loose", strict=False)
        items = _filter_relevant(_dedupe(raw), ontology, target_dt)

    # Pass 3: Tier2 fallback
    if not items:
        raw += _search_stage(ontology, alert_date, SOURCES_TIER2, "Tier2-Fallback", strict=False)
        items = _filter_relevant(_dedupe(raw), ontology, target_dt)

    if not items:
        return "No relevant D/D-1 news found for the specified ontology and date window."

    if USE_LLM_RERANK:
        items = _llm_rerank(items, ontology)
        if not items:
            return "No relevant D/D-1 news found for the specified ontology and date window (after LLM filter)."

    # rank & select top k
    items.sort(key=lambda x: (_score(x, ontology), x["_pub_dt"] or datetime.min), reverse=True)
    top = items[:k]

    # format output
    lines = []
    for i, it in enumerate(top, 1):
        title = (it.get("title") or "").strip()
        snippet = (it.get("content") or "").strip().replace("\n", " ")
        if len(snippet) > 220: snippet = snippet[:217] + "…"
        url = it.get("url") or ""
        domain = url.split("/")[2] if "://" in url else url
        tier = it.get("_tier", "Unknown")
        dt_str = it["_pub_dt"].strftime("%d %b %Y") if it.get("_pub_dt") else "Unknown date"
        header = f"{i}. [{dt_str}] [{domain}] [{tier}]"
        lines.append(f"{header} {title} — {snippet} ({url})" if title else f"{header} {snippet} ({url})")
    return "\n".join(lines)

# =========================
# REASONER
# =========================
REASONING_PROMPT = """
You are a market surveillance analyst.

Analyze the following news and determine if it could explain a spike in trading alerts for {ontology} on {alert_date}.

Instructions:
1) ONLY use the information present in the news items; do not invent facts.
2) If the news clearly indicates volatility, price/volume shock, major policy/geopolitical event, or supply/demand shock, explain briefly how that could increase alerts.
3) If the news is neutral or does not suggest abnormal activity, explicitly state that it likely does NOT explain a spike.
4) Prefer precision over generic language. 1–2 sentences max.

News:
{news_summary}

Output:
"""

def run_reasoning_agent(news_summary: str, ontology: str, alert_date: str) -> str:
    if news_summary.startswith("No relevant"):
        return "No D/D-1 news available; insufficient evidence to link this alert to external news."
    prompt = REASONING_PROMPT.format(
        news_summary=news_summary,
        ontology=ontology,
        alert_date=alert_date
    )
    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=120,
        messages=[
            {"role": "system", "content": "You are a precise, cautious market surveillance analyst."},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content.strip()

# =========================
# MAIN
# =========================
def main():
    df = read_alerts(INPUT_PATH)
    print("Input shape:", df.shape)
    print(df.head())

    news_list, reasoning_list = [], []
    for _, row in df.iterrows():
        ontology   = str(row["cdh_asset_ontology_name"])
        alert_date = row["Date"].strftime("%d %B %Y")
        row['Date'] = '2025-08-06'

        news_summary = run_news_agent(ontology, alert_date)
        reasoning    = run_reasoning_agent(news_summary, ontology, alert_date)

        print(f"\n--- {ontology} | {alert_date} ---")
        print("NEWS:\n", news_summary)
        print("REASONING:\n", reasoning)

        news_list.append(news_summary)
        reasoning_list.append(reasoning)

    df["News_Summary"]  = news_list
    df["LLM_Reasoning"] = reasoning_list

    write_alerts(df, OUTPUT_PATH)
    print("✅ Alert reasoning written to", OUTPUT_PATH)

if __name__ == "__main__":
    main()


Input shape: (2, 3)
   Alert_ID cdh_asset_ontology_name       Date
0      1001               Crude Oil 2024-05-21
1      1002                    Gold 2025-08-07

--- Crude Oil | 21 May 2024 ---
NEWS:
 1. [21 May 2024] [www.reuters.com] [Tier1-Loose] Oil falls 1% on sticky US inflation, dampened geopolitical ... — U.S. crude oil and gasoline inventories rose last week, while distillates fell, according to market sources citing American Petroleum Institute (https://www.reuters.com/business/energy/oil-prices-fall-fear-high-us-interest-rates-depressing-demand-2024-05-21/)
2. [21 May 2024] [www.reuters.com] [Tier1-Loose] Brent oil market structure weakens as tightness concern ... — The structure of the benchmark Brent crude oil futures market fell on Tuesday to its weakest since February, another indication that concern about tight supply (https://www.reuters.com/markets/commodities/brent-oil-market-structure-weakens-tightness-concern-eases-2024-05-21/)
3. [21 May 2024] [www.cnbc.com] [Tier